Install Libraries

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 77.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install transformers datasets seqeval scikit-learn jsonlines

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=1f033441d8039960e9acb66b889d8ebd723ed4cec48e4d2a492ccc96b92c743d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Imports

In [3]:
import json
import jsonlines
import numpy as np

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

Label Definitions

In [4]:
labels = [
    "O",
    "B-TOPIC","I-TOPIC",
    "B-METHOD","I-METHOD",
    "B-CONCEPT","I-CONCEPT",
    "B-ALGORITHM","I-ALGORITHM",
    "B-OTHER","I-OTHER"
]

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

Load Dataset

In [6]:
data = []

with jsonlines.open("OS-cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

Convert Entities → BIO Labels

In [7]:
def convert_to_tokens(sample):

    text = sample["input"]
    entities = json.loads(sample["output"])

    tokens = text.split()
    tags = ["O"] * len(tokens)

    for ent in entities:

        ent_text = ent["entity"]
        label = ent["label"]

        ent_tokens = ent_text.split()

        for i in range(len(tokens)):

            if tokens[i:i+len(ent_tokens)] == ent_tokens:

                tags[i] = "B-" + label

                for j in range(1,len(ent_tokens)):
                    tags[i+j] = "I-" + label

    return {"tokens":tokens,"ner_tags":tags}

Prepare Dataset

In [8]:
processed = []

for sample in data:
    processed.append(convert_to_tokens(sample))

dataset = Dataset.from_list(processed)

Train/Test Split (90/10)

In [9]:
train_test = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = train_test["train"]
test_dataset = train_test["test"]

Load SciBERT

In [10]:
model_name = "allenai/scibert_scivocab_cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

Tokenization

In [11]:
MAX_LEN = 512

def tokenize(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=MAX_LEN,        # ✅ truncate sequences to 512
        is_split_into_words=True,
    )

    word_ids = tokenized.word_ids()
    labels_ids = []

    for word_id in word_ids:
        if word_id is None:
            labels_ids.append(-100)
        else:
            label = example["ner_tags"][word_id].upper()
            labels_ids.append(label2id.get(label, label2id["O"]))

    tokenized["labels"] = labels_ids
    return tokenized

In [12]:
train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/1249 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Training Arguments

In [13]:
training_args = TrainingArguments(

    output_dir="scibert_ner",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Metrics Function

In [14]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):

        temp_pred = []
        temp_lab = []

        for p_i,l_i in zip(pred,lab):

            if l_i != -100:
                temp_pred.append(id2label[p_i])
                temp_lab.append(id2label[l_i])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {

        "precision": precision_score(true_labels,true_predictions),
        "recall": recall_score(true_labels,true_predictions),
        "f1": f1_score(true_labels,true_predictions)
    }

Trainer

In [15]:
from transformers import DataCollatorForTokenClassification

# This will pad both input_ids and labels to the longest sequence in the batch
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

# Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,  # ✅ ensures consistent batch lengths
    compute_metrics=compute_metrics
)

Train Model

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.100143,0.689169,0.687437,0.688302
2,No log,0.076815,0.717181,0.818090,0.764319
3,No log,0.067566,0.767287,0.808543,0.787375


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=471, training_loss=0.11637662170798915, metrics={'train_runtime': 316.7647, 'train_samples_per_second': 11.829, 'train_steps_per_second': 1.487, 'total_flos': 642927558985998.0, 'train_loss': 0.11637662170798915, 'epoch': 3.0})

Evaluate Model

In [17]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import accuracy_score

# Get predictions (this works regardless of evaluate hooks)
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    temp_pred, temp_lab = [], []
    for p_i, l_i in zip(pred_seq, label_seq):
        if l_i != -100:  # ignore padding
            temp_pred.append(id2label[p_i])
            temp_lab.append(id2label[l_i])
    true_predictions.append(temp_pred)
    true_labels.append(temp_lab)

# Compute metrics
micro_precision = precision_score(true_labels, true_predictions)
micro_recall = recall_score(true_labels, true_predictions)
micro_f1 = f1_score(true_labels, true_predictions)

report = classification_report(true_labels, true_predictions, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]

# Exact accuracy
flat_true, flat_pred = [], []
for t_seq, p_seq in zip(true_labels, true_predictions):
    flat_true.extend(t_seq)
    flat_pred.extend(p_seq)
exact_accuracy = accuracy_score(flat_true, flat_pred)

# Partial accuracy
partial_matches = 0
total_tokens = 0
for t_seq, p_seq in zip(true_labels, true_predictions):
    for t, p in zip(t_seq, p_seq):
        total_tokens += 1
        if t == p:
            partial_matches += 1
        elif t != "O" and p != "O" and t.split("-")[-1] == p.split("-")[-1]:
            partial_matches += 0.5
partial_accuracy = partial_matches / total_tokens

# Print metrics
print("\n===== Evaluation Metrics =====\n")
print("Micro Precision :", round(micro_precision,4))
print("Micro Recall    :", round(micro_recall,4))
print("Micro F1        :", round(micro_f1,4))
print("Macro F1        :", round(macro_f1,4))
print("Exact Accuracy  :", round(exact_accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))
print("\nDetailed Classification Report:\n")
print(classification_report(true_labels, true_predictions))


===== Evaluation Metrics =====

Micro Precision : 0.7673
Micro Recall    : 0.8085
Micro F1        : 0.7874
Macro F1        : 0.7048
Exact Accuracy  : 0.9796
Partial Accuracy: 0.9798

Detailed Classification Report:

              precision    recall  f1-score   support

   ALGORITHM       0.83      0.84      0.84       121
     CONCEPT       0.74      0.86      0.80      1081
      METHOD       0.80      0.18      0.29       114
       OTHER       0.89      0.83      0.86       407
       TOPIC       0.69      0.80      0.74       267

   micro avg       0.77      0.81      0.79      1990
   macro avg       0.79      0.70      0.70      1990
weighted avg       0.77      0.81      0.78      1990



In [18]:
!rm -rf /content/scibert_ner